In [ ]:
class MagicList:

    def __init__(self):
        self.data = list()

    def __getitem__(self, key):
        return self.data[key]
    
    def __setitem__(self, key, value):
        self.data[key] = value

    def __len__(self):
        return len(self.data)
    
    def append(self, item):
        self.data.append(item)

ml = MagicList()
ml.append(1); ml.append(2); ml.append(3)
print(ml[0])        # 1
ml[1] = 99
print(ml[1:3])      # [99, 3]
print(len(ml))      # 3  
        

In [ ]:
class Shape:

    def __init__(self, color):
        self._color = color

    def get_color(self):
        return self._color
    
    def set_color(self, color):
        self._color = color

class Circle(Shape):

    def __init__(self, color, radius):
        super().__init__(color)
        if not isinstance(radius, (int, float)) or radius <= 0:
            raise AttributeError("Raduis is not valid")
        
        self.__radius = radius

    @property
    def area(self):
        import math
        return math.pi * self.__radius ** 2
    
    def __str__(self):
        return f"Circle with raduis {self.__radius}"
    
    def _protected_method(self):
        pass

c = Circle("red", 5)
print(c.get_color())   # red
print(c.area)          # 78.5398...
c._protected_method()  # сработает, но IDE предупредит
print(c)


In [ ]:
class AddAttr:

    def __init__(self, name, value):
        self.name = name
        self.value = value

    def __call__(self, cls, *args, **kwds):
        setattr(cls, self.name, self.value)
        return cls

@AddAttr("version", 1.0)
class MyClass:
    pass

print(MyClass.version)   # 1.0    

In [ ]:
class Range:

    def __init__(self, min_val=-273.15, max_val=1000):
        self.min_val = min_val
        self.max_val = max_val
        self._name = None  # будет хранить имя атрибута

    def __set_name__(self, owner, name):
        self._name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self

        return instance.__dict__.get(self._name)

    def __set__(self, instance, value):
        
        if not (self.min_val <= value <= self.max_val):
            raise ValueError(
                f"Значение {value} не входит в допустимый диапазон "
                f"[{self.min_val}, {self.max_val}]"
            )
        
        instance.__dict__[self._name] = value


class Temperature:
    celsius = Range()


# Пример использования
if __name__ == "__main__":
    t = Temperature()
    t.celsius = 25        # OK
    print(t.celsius)      # 25
    try:
        t.celsius = -300  # Вызовет ValueError
    except ValueError as e:
        print(f"Ошибка: {e}")

In [ ]:
import time

class CachedFunction:

    def __new__(cls, func):
        instance = super().__new__(cls)
        return instance

    def __init__(self, func):
        self.func = func
        self.cache = {}

    def __call__(self, *args, **kwargs):
        key = (args, tuple(sorted(kwargs.items())))
        if key in self.cache:
            print("Из кэша")
            return self.cache[key]
        else:
            print("Вычисляем")
            result = self.func(*args, **kwargs)
            self.cache[key] = result
            return result



@CachedFunction
def slow_square(x):
    time.sleep(1)  # имитация долгого вычисления
    return x * x

print(slow_square(4))   # Вычисляем -> 16 (с задержкой)
print(slow_square(4))   # Из кэша -> 16 (мгновенно)
print(slow_square(5))   # Вычисляем -> 25 (задержка)
print(slow_square(5))   # Из кэша -> 25

In [ ]:
class SortFilterDecorator:

    def __init__(self, reverse=False, numeric_only=False):
        self.reverse = reverse
        self.numeric_only = numeric_only

    def __call__(self, func):
        def wrapper(lst, *args, **kwargs):
            processed = lst.copy()
            if self.numeric_only:
                processed = [x for x in processed if isinstance(x, (int, float))]
            processed.sort(reverse=self.reverse)
            return func(processed, *args, **kwargs)
        return wrapper

@SortFilterDecorator(reverse=True, numeric_only=True)
def list_to_joined_string(lst):
    return ', '.join(str(item) for item in lst)



data = [1, 'hello', 3.5, 'world', 2, 7, 'abc']
print(list_to_joined_string(data))  # Вывод: 7, 3.5, 2, 1


@SortFilterDecorator(reverse=False, numeric_only=False)
def list_to_joined_string2(lst):
    return ', '.join(str(item) for item in lst)

data2 = ['banana', 'apple', 'cherry']
print(list_to_joined_string2(data2))  # Вывод: apple, banana, cherry